# Nowcasting Model: Comparing Baseline, Lag12, and Lag12 + Load Factor

Notebook 16 showed that adding `delay_rate_lag12` improved the nowcasting model. This notebook keeps the same model setup and adds one extra feature from notebook 12: raw `load_factor` (overall/industry-aggregate monthly passenger load factor). This notebook compares the performance of the following models:
- the previous nowcasting baseline (notebook 8b)
- baseline + lag12
- baseline + lag12 + `load_factor`

## Reference

| Source | Ridge R2 | RF R2 | XGB F1 |
|--------|----------|-------|--------|
| Nowcasting 8b (baseline) | 0.462 | 0.486 | 0.733 |
| Nowcasting 16 (+lag12) | 0.520 | 0.529 | 0.738 |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation

Same filtering and modelling setup as notebook 16, with `load_factor` merged by `year_month` following notebook 12.

In [ ]:
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

# Load BITRE Domestic airlines sheet and extract monthly industry load factor (raw only)
import os
activity_candidates = [
    '../data/raw/monthly-airline-performance-october-2025.xlsx',
    '../data/raw/monthly-airline-performance-november-2025.xlsx',
]
ACTIVITY_FILE = next((f for f in activity_candidates if os.path.exists(f)), None)
if ACTIVITY_FILE is None:
    raise FileNotFoundError('Monthly airline performance file not found in expected locations.')
SHEET_NAME = 'Domestic airlines'

df_activity = pd.read_excel(ACTIVITY_FILE, sheet_name=SHEET_NAME, header=None, skiprows=8)
df_activity.columns = [
    'year', 'month_name', 'hours_flown', 'aircraft_km_flown_000', 'aircraft_departures',
    'total_rev_pax_ud', 'freight_tonnes_ud', 'mail_tonnes_ud',
    'total_rev_pax_tob', 'total_rev_pax_tob_inc_intl',
    'freight_tonnes_tob', 'mail_tonnes_tob',
    'total_rpk_000', 'pax_tonne_km_000', 'freight_tonne_km_000',
    'mail_tonne_km_000', 'total_tonne_km_000',
    'available_seat_km_000', 'available_tonne_km_000', 'available_seats_000',
    'pax_load_factor_pct', 'weight_load_factor_pct',
    'total_charter_pax_tob', 'charter_aircraft_departures'
]

df_activity['year_numeric'] = pd.to_numeric(df_activity['year'], errors='coerce')
valid_months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'June',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

df_act = df_activity[
    df_activity['year_numeric'].notna() &
    df_activity['month_name'].isin(valid_months)
].copy()

df_act['year'] = df_act['year_numeric'].astype(int)
df_act = df_act.drop(columns=['year_numeric'])
df_act['month_name'] = df_act['month_name'].replace({'June': 'Jun'})

month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
             'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
df_act['month_num'] = df_act['month_name'].map(month_map)
df_act['year_month'] = df_act['year'].astype(str) + '-' + df_act['month_num'].astype(str).str.zfill(2)
df_act['load_factor'] = pd.to_numeric(df_act['pax_load_factor_pct'], errors='coerce') / 100.0

# Match notebook 12 filtering choices
covid_mask = (df_act['year'] == 2020) & (df_act['month_num'] >= 4)
df_act = df_act[~covid_mask].copy()
df_act = df_act[df_act['year'] >= 2009].copy()

df_lf = df_act[['year_month', 'load_factor']].groupby('year_month', as_index=False).mean()
df = df.merge(df_lf, on='year_month', how='left')

matched = df['load_factor'].notna().sum()
match_rate = matched / len(df) * 100

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print('Merged shape: {}'.format(df.shape))
print('Load factor source: {}'.format(ACTIVITY_FILE))
print('Load factor match rate: {:.1f}% ({}/{})'.format(match_rate, matched, len(df)))

In [ ]:
# Filter (same as 8b)
volume_threshold = 40
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean()
high_volume_ar = airline_route_volume[airline_route_volume >= volume_threshold].index.tolist()
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney', 'Perth_Brisbane']
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()
print("Records after filtering: {}".format(len(df_filtered)))

## 2. Feature Engineering

In [ ]:
# Standard lag features (same as 8b)
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# NEW: lag12
df_filtered['delay_rate_lag12'] = df_filtered.groupby('airline_route')['delay_rate'].shift(12)

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Weather features (same as 8b)
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['extreme_weather_days_total'] = df_filtered['extreme_weather_days_dep'] + df_filtered['extreme_weather_days_arr']
rainy_max = df_filtered['rainy_days_arr'].max()
temp_vol_max = df_filtered['temp_volatility_total'].max()
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / rainy_max)
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / temp_vol_max)

print("Airlines: {}".format(len(airline_cols)))
print("Routes:   {}".format(len(route_cols)))

## 3. Correlation Analysis

In [ ]:
corr_cols = ['delay_rate', 'delay_rate_lag1', 'delay_rate_lag12', 'delay_rate_gradient',
             'rainy_days_arr_exp', 'temp_volatility_total_exp', 'extreme_weather_days_total', 'load_factor']
corrs = df_filtered[corr_cols].corr().loc['delay_rate'].drop('delay_rate')

print('Correlation with delay_rate:')
print('-' * 45)
for feat, val in corrs.sort_values(ascending=False).items():
    print('  {:<35} {:.4f}'.format(feat, val))

In [ ]:
# Scatter: lag12 and load_factor vs target
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

valid = df_filtered.dropna(subset=['delay_rate', 'delay_rate_lag12', 'load_factor'])

ax = axes[0]
ax.scatter(valid['delay_rate_lag12'], valid['delay_rate'], alpha=0.3, s=10)
ax.plot([0, 0.6], [0, 0.6], 'r--')
ax.set_xlabel('delay_rate_lag12')
ax.set_ylabel('delay_rate')
ax.set_title('Lag12 vs Target (r = {:.3f})'.format(corrs['delay_rate_lag12']))
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 0.6)

ax = axes[1]
ax.scatter(valid['load_factor'], valid['delay_rate'], alpha=0.3, s=10)
ax.set_xlabel('load_factor')
ax.set_ylabel('delay_rate')
ax.set_title('Load Factor vs Target (r = {:.3f})'.format(corrs['load_factor']))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Data Loss Comparison

In [ ]:
df_baseline = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
df_with_lag12 = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient', 'delay_rate_lag12']).copy()
df_with_lag12_lf = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient', 'delay_rate_lag12', 'load_factor']).copy()

print('Data availability:')
print('  Baseline (8b):            {:,} rows'.format(len(df_baseline)))
print('  With lag12:               {:,} rows'.format(len(df_with_lag12)))
print('  With lag12 + load_factor: {:,} rows'.format(len(df_with_lag12_lf)))
print('  Rows lost (lag12):        {:,} ({:.1f}%)'.format(
    len(df_baseline) - len(df_with_lag12),
    (len(df_baseline) - len(df_with_lag12)) / len(df_baseline) * 100
))
print('  Rows lost (+load_factor): {:,} ({:.1f}%)'.format(
    len(df_with_lag12) - len(df_with_lag12_lf),
    (len(df_with_lag12) - len(df_with_lag12_lf)) / len(df_with_lag12) * 100 if len(df_with_lag12) else 0.0
))

## 5. Train/Val/Test Split

Use the lag12+load_factor-available subset for fair comparison across all three models.

In [ ]:
df_clean = df_with_lag12_lf.copy()

train_mask = (((df_clean['year'] <= 2017)) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print('Train: {:,}'.format(train_mask.sum()))
print('Val:   {:,}'.format(val_mask.sum()))
print('Test:  {:,}'.format(test_mask.sum()))

In [ ]:
# Feature sets
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
weather_features = ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp', 'extreme_weather_days_total']
holiday_features = ['n_public_holidays_total', 'pct_school_holiday']

# Baseline: 8b feature set
features_baseline = base_features + weather_features + holiday_features

# Lag12 only: 8b + lag12
features_with_lag12 = base_features + weather_features + ['delay_rate_lag12'] + holiday_features

# Lag12 + load_factor (raw only): 8b + lag12 + load_factor
features_with_lag12_lf = base_features + weather_features + ['delay_rate_lag12', 'load_factor'] + holiday_features

print('Baseline features:            {}'.format(len(features_baseline)))
print('With lag12:                   {}'.format(len(features_with_lag12)))
print('With lag12 + load_factor:     {}'.format(len(features_with_lag12_lf)))

## 6. Model Comparison

In [ ]:
# Tune Ridge alpha on validation set (RMSE)
alphas = np.logspace(-2, 4, 13)

def tune_ridge_alpha(df, features, train_mask, val_mask, alphas):
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    y_train = df.loc[train_mask, 'delay_rate'].values
    y_val = df.loc[val_mask, 'delay_rate'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    rows = []
    for a in alphas:
        ridge = Ridge(alpha=a)
        ridge.fit(X_train_scaled, y_train)
        pred = ridge.predict(X_val_scaled)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        r2 = r2_score(y_val, pred)
        rows.append({'alpha': a, 'rmse': rmse, 'r2': r2})

    results = pd.DataFrame(rows).sort_values('rmse')
    best_alpha = float(results.iloc[0]['alpha'])
    return best_alpha, results

ridge_alpha_baseline, ridge_tune_baseline = tune_ridge_alpha(
    df_clean, features_baseline, train_mask, val_mask, alphas
)
ridge_alpha_lag12, ridge_tune_lag12 = tune_ridge_alpha(
    df_clean, features_with_lag12, train_mask, val_mask, alphas
)
ridge_alpha_lag12_lf, ridge_tune_lag12_lf = tune_ridge_alpha(
    df_clean, features_with_lag12_lf, train_mask, val_mask, alphas
)

print('Best alpha (baseline):', ridge_alpha_baseline)
print('Best alpha (lag12):', ridge_alpha_lag12)
print('Best alpha (lag12 + load_factor):', ridge_alpha_lag12_lf)

# Plot validation R2 vs alpha for all three variants
plt.figure(figsize=(7, 4))
for label, tune_df in [
    ('Baseline', ridge_tune_baseline),
    ('Lag12', ridge_tune_lag12),
    ('Lag12 + LF', ridge_tune_lag12_lf),
]:
    d = tune_df.sort_values('alpha')
    plt.semilogx(d['alpha'], d['r2'], marker='o', label=label)

plt.xlabel('Alpha')
plt.ylabel('Validation R2')
plt.title('Ridge Alpha Tuning (Three Variants)')
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.legend()
plt.show()

In [ ]:
def evaluate_model(df, features, train_mask, val_mask, test_mask, name, ridge_alpha=100):
    """Train and evaluate Ridge, RF, and XGBoost."""
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    X_test = df.loc[test_mask, features].values

    y_train = df.loc[train_mask, 'delay_rate'].values
    y_val = df.loc[val_mask, 'delay_rate'].values
    y_test = df.loc[test_mask, 'delay_rate'].values

    y_train_clf = df.loc[train_mask, 'is_high_delay'].values
    y_val_clf = df.loc[val_mask, 'is_high_delay'].values
    y_test_clf = df.loc[test_mask, 'is_high_delay'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    results = {'name': name}

    # Ridge
    ridge = Ridge(alpha=ridge_alpha)
    ridge.fit(X_train_scaled, y_train)
    ridge_pred = ridge.predict(X_test_scaled)
    results['ridge_r2'] = r2_score(y_test, ridge_pred)
    results['ridge_rmse'] = np.sqrt(mean_squared_error(y_test, ridge_pred))

    # Random Forest Regression
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    results['rf_r2'] = r2_score(y_test, rf_pred)
    results['rf_rmse'] = np.sqrt(mean_squared_error(y_test, rf_pred))
    results['rf_model'] = rf

    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                     min_child_weight=5, random_state=42, n_jobs=-1)
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_pred = xgb_clf.predict(X_test)
        results['xgb_f1'] = f1_score(y_test_clf, xgb_pred)
    else:
        results['xgb_f1'] = np.nan

    return results, features


In [ ]:
print('Training models...')
print()

results_baseline, _ = evaluate_model(
    df_clean, features_baseline, train_mask, val_mask, test_mask,
    'Baseline (8b)', ridge_alpha=ridge_alpha_baseline
)
results_lag12, _ = evaluate_model(
    df_clean, features_with_lag12, train_mask, val_mask, test_mask,
    'Lag12 Only', ridge_alpha=ridge_alpha_lag12
)
results_lag12_lf, _ = evaluate_model(
    df_clean, features_with_lag12_lf, train_mask, val_mask, test_mask,
    'Lag12 + Load Factor', ridge_alpha=ridge_alpha_lag12_lf
)

print('=' * 80)
print('RESULTS COMPARISON')
print('=' * 80)
print()
header = '{:<20} {:>10} {:>12} {:>10} {:>12} {:>10}'.format(
    'Model', 'Ridge R2', 'Ridge RMSE', 'RF R2', 'RF RMSE', 'XGB F1'
)
print(header)
print('-' * 82)

for r in [results_baseline, results_lag12, results_lag12_lf]:
    xgb_str = '{:.4f}'.format(r['xgb_f1']) if not np.isnan(r['xgb_f1']) else 'N/A'
    print('{:<20} {:>10.4f} {:>12.4f} {:>10.4f} {:>12.4f} {:>10}'.format(
        r['name'], r['ridge_r2'], r['ridge_rmse'], r['rf_r2'], r['rf_rmse'], xgb_str
    ))

print()
print('Delta (Lag12 - Baseline):')
print('  Ridge R2:   {:+.4f}'.format(results_lag12['ridge_r2'] - results_baseline['ridge_r2']))
print('  Ridge RMSE: {:+.4f}'.format(results_lag12['ridge_rmse'] - results_baseline['ridge_rmse']))
print('  RF R2:      {:+.4f}'.format(results_lag12['rf_r2'] - results_baseline['rf_r2']))
print('  RF RMSE:    {:+.4f}'.format(results_lag12['rf_rmse'] - results_baseline['rf_rmse']))
if HAS_XGB:
    print('  XGB F1:     {:+.4f}'.format(results_lag12['xgb_f1'] - results_baseline['xgb_f1']))

print()
print('Delta (Lag12+LF - Lag12):')
print('  Ridge R2:   {:+.4f}'.format(results_lag12_lf['ridge_r2'] - results_lag12['ridge_r2']))
print('  Ridge RMSE: {:+.4f}'.format(results_lag12_lf['ridge_rmse'] - results_lag12['ridge_rmse']))
print('  RF R2:      {:+.4f}'.format(results_lag12_lf['rf_r2'] - results_lag12['rf_r2']))
print('  RF RMSE:    {:+.4f}'.format(results_lag12_lf['rf_rmse'] - results_lag12['rf_rmse']))
if HAS_XGB:
    print('  XGB F1:     {:+.4f}'.format(results_lag12_lf['xgb_f1'] - results_lag12['xgb_f1']))

## 7. Feature Importance

In [ ]:
rf_model = results_lag12_lf['rf_model']
importance_df = pd.DataFrame({
    'Feature': features_with_lag12_lf,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

new_features = {'delay_rate_lag12', 'load_factor'}

print('Feature Importance (RF: Lag12 + Load Factor variant):')
print('-' * 62)
for _, row in importance_df.head(15).iterrows():
    marker = ' <-- NEW' if row['Feature'] in new_features else ''
    print('  {:<35} {:.4f}{}'.format(row['Feature'], row['Importance'], marker))

lag12_rank = list(importance_df['Feature']).index('delay_rate_lag12') + 1
lag12_importance = importance_df[importance_df['Feature'] == 'delay_rate_lag12']['Importance'].values[0]
lf_rank = list(importance_df['Feature']).index('load_factor') + 1
lf_importance = importance_df[importance_df['Feature'] == 'load_factor']['Importance'].values[0]

print()
print('Lag12 rank: {} (importance: {:.4f})'.format(lag12_rank, lag12_importance))
print('Load factor rank: {} (importance: {:.4f})'.format(lf_rank, lf_importance))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

top_features = importance_df.head(12)
y_pos = np.arange(len(top_features))
colors = [
    '#e74c3c' if f == 'delay_rate_lag12'
    else '#f39c12' if f == 'load_factor'
    else 'steelblue'
    for f in top_features['Feature']
]

ax.barh(y_pos, top_features['Importance'], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_features['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Feature Importances: Nowcasting with Lag12 + Load Factor')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 8. Route-Level Performance

In [ ]:
# Predictions for route-level analysis (Ridge)
# Baseline Ridge
scaler_b = StandardScaler()
X_train_b = scaler_b.fit_transform(df_clean.loc[train_mask, features_baseline].values)
X_test_b = scaler_b.transform(df_clean.loc[test_mask, features_baseline].values)
ridge_b = Ridge(alpha=ridge_alpha_baseline)
ridge_b.fit(X_train_b, df_clean.loc[train_mask, 'delay_rate'].values)

# Lag12 Ridge
scaler_l = StandardScaler()
X_train_l = scaler_l.fit_transform(df_clean.loc[train_mask, features_with_lag12].values)
X_test_l = scaler_l.transform(df_clean.loc[test_mask, features_with_lag12].values)
ridge_l = Ridge(alpha=ridge_alpha_lag12)
ridge_l.fit(X_train_l, df_clean.loc[train_mask, 'delay_rate'].values)

# Lag12 + load_factor Ridge
scaler_lf = StandardScaler()
X_train_lf = scaler_lf.fit_transform(df_clean.loc[train_mask, features_with_lag12_lf].values)
X_test_lf = scaler_lf.transform(df_clean.loc[test_mask, features_with_lag12_lf].values)
ridge_lf = Ridge(alpha=ridge_alpha_lag12_lf)
ridge_lf.fit(X_train_lf, df_clean.loc[train_mask, 'delay_rate'].values)

df_test = df_clean[test_mask].copy()
df_test['baseline_pred'] = ridge_b.predict(X_test_b)
df_test['lag12_pred'] = ridge_l.predict(X_test_l)
df_test['lag12_lf_pred'] = ridge_lf.predict(X_test_lf)

print('Route-Level Performance (Ridge R2):')
print('=' * 94)
header = '{:<20} {:>10} {:>10} {:>12} {:>10} {:>10}'.format(
    'Route', 'Baseline', 'Lag12', 'Lag12+LF', 'dL12', 'dLF'
)
print(header)
print('-' * 84)

route_results = []
for route in sorted(df_test['route'].unique()):
    rd = df_test[df_test['route'] == route]
    r2_b = r2_score(rd['delay_rate'], rd['baseline_pred'])
    r2_l = r2_score(rd['delay_rate'], rd['lag12_pred'])
    r2_lf = r2_score(rd['delay_rate'], rd['lag12_lf_pred'])
    d_l12 = r2_l - r2_b
    d_lf = r2_lf - r2_l
    print('{:<20} {:>10.4f} {:>10.4f} {:>12.4f} {:>+10.4f} {:>+10.4f}'.format(
        route, r2_b, r2_l, r2_lf, d_l12, d_lf
    ))
    route_results.append({
        'Route': route,
        'Baseline': r2_b,
        'With_Lag12': r2_l,
        'With_Lag12_LF': r2_lf,
        'Delta_Lag12': d_l12,
        'Delta_LF': d_lf,
    })

route_df = pd.DataFrame(route_results)
print('-' * 84)
overall_b = r2_score(df_test['delay_rate'], df_test['baseline_pred'])
overall_l = r2_score(df_test['delay_rate'], df_test['lag12_pred'])
overall_lf = r2_score(df_test['delay_rate'], df_test['lag12_lf_pred'])
print('{:<20} {:>10.4f} {:>10.4f} {:>12.4f} {:>+10.4f} {:>+10.4f}'.format(
    'OVERALL', overall_b, overall_l, overall_lf, overall_l - overall_b, overall_lf - overall_l
))

In [ ]:
# Visualize route-level deltas (grouped)
fig, ax = plt.subplots(figsize=(14, 5))

routes = route_df['Route'].values
x = np.arange(len(routes))
width = 0.38

delta_lag12 = route_df['Delta_Lag12'].values
delta_lf = route_df['Delta_LF'].values

ax.bar(x - width / 2, delta_lag12, width, color='steelblue', alpha=0.85, label='Lag12 - Baseline')
ax.bar(x + width / 2, delta_lf, width, color='#f39c12', alpha=0.85, label='Lag12+LF - Lag12')

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xticks(x)
route_labels = [r.replace('_', ' to ') for r in routes]
ax.set_xticklabels(route_labels, rotation=45, ha='right')
ax.set_ylabel('Delta R2')
ax.set_title('Route-Level Incremental Impact: Lag12 then Load Factor')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()

plt.tight_layout()
plt.show()

improved_lag12 = (route_df['Delta_Lag12'] > 0).sum()
improved_lf = (route_df['Delta_LF'] > 0).sum()
print('{} of {} routes improved with lag12 (vs baseline)'.format(improved_lag12, len(route_df)))
print('{} of {} routes improved with load_factor (vs lag12)'.format(improved_lf, len(route_df)))

## 9. Summary and Observations

In [ ]:
print('=' * 84)
print('THREE-WAY NOWCASTING COMPARISON: BASELINE vs LAG12 vs LAG12 + LOAD FACTOR')
print('=' * 84)
print()

delta_l12_ridge = results_lag12['ridge_r2'] - results_baseline['ridge_r2']
delta_l12_rf = results_lag12['rf_r2'] - results_baseline['rf_r2']
delta_l12_xgb = results_lag12['xgb_f1'] - results_baseline['xgb_f1'] if HAS_XGB else 0

delta_lf_ridge = results_lag12_lf['ridge_r2'] - results_lag12['ridge_r2']
delta_lf_rf = results_lag12_lf['rf_r2'] - results_lag12['rf_r2']
delta_lf_xgb = results_lag12_lf['xgb_f1'] - results_lag12['xgb_f1'] if HAS_XGB else 0

print('PERFORMANCE (Test Set):')
print('  Baseline          Ridge {:.4f} | RF {:.4f} | XGB {:.4f}'.format(
    results_baseline['ridge_r2'], results_baseline['rf_r2'], results_baseline['xgb_f1']
))
print('  Lag12 only        Ridge {:.4f} | RF {:.4f} | XGB {:.4f}'.format(
    results_lag12['ridge_r2'], results_lag12['rf_r2'], results_lag12['xgb_f1']
))
print('  Lag12 + LF        Ridge {:.4f} | RF {:.4f} | XGB {:.4f}'.format(
    results_lag12_lf['ridge_r2'], results_lag12_lf['rf_r2'], results_lag12_lf['xgb_f1']
))
print()

print('INCREMENTAL DELTAS:')
print('  Lag12 - Baseline:')
print('    Ridge R2 {:+.4f}, RF R2 {:+.4f}, XGB F1 {:+.4f}'.format(delta_l12_ridge, delta_l12_rf, delta_l12_xgb))
print('  (Lag12 + LF) - Lag12:')
print('    Ridge R2 {:+.4f}, RF R2 {:+.4f}, XGB F1 {:+.4f}'.format(delta_lf_ridge, delta_lf_rf, delta_lf_xgb))
print()



### Observations

- Adding load factor overall impacts the regression models negatively: very small improvement to Ridge model but a noticeable penalty to RF model
- In contrast, adding load factor improves XGB model (classification)
- However, BITRE releases the load factor data much later than to the on-time-performance
    - At the time of writing, it is approaching the end of February but BITRE has not released the January data
    - Thus, load factor is therefore excluded from the production model, despite its positive impact on the XGBoost classification model.

## 10. Next Step

The next step is to add 12-month lagged delay rate to the neural-network forecasting model. This is motivated by the positive impact of this feature in other models (notebooks 15a and 16).